# Ecobici — 00. Datos crudos y limpieza

Carga de Datos y limpieza

- **Mes principal:** septiembre 2026 (octubre 2026 aún no aparece publicado).
- **Ventana para escenarios:** octubre 2025 – septiembre 2026.
- **Subred:** 10 estaciones del corredor Buenavista–Reforma.

## 1. Configuración

Definimos las rutas, las URLs de los 12 meses (las mismas de `pipeline_datos_ecobicis.ipynb`) y las estaciones de la subred. 

In [2]:
from pathlib import Path
import json

import pandas as pd
import requests

pd.set_option("display.max_columns", None)

DIR_TRIPS = Path("data/raw/trips")
RUTA_ESTACIONES = Path("data/raw/station_information.json")

MES_PRINCIPAL = "2026-09"

BASE = "https://ecobici.cdmx.gob.mx/wp-content/uploads"
URLS = {
    "2025-10": f"{BASE}/2025/11/2025-10-1.csv",
    "2025-11": f"{BASE}/2025/12/2025-11.csv",
    "2025-12": f"{BASE}/2026/01/2025-12.csv",
    "2026-01": f"{BASE}/2026/02/2026-01.csv",
    "2026-02": f"{BASE}/2026/03/2026-02.csv",
    "2026-03": f"{BASE}/2026/04/2026-03.csv",
    "2026-04": f"{BASE}/2026/05/2026-04.csv",
    "2026-05": f"{BASE}/2026/06/2026-05.csv",
    "2026-06": f"{BASE}/2026/07/public_data_web_2026-06.csv",
    "2026-07": f"{BASE}/2026/08/public_data_web_2026-07.csv",
    "2026-08": f"{BASE}/2026/09/public_data_web_2026-08_2.csv",
    "2026-09": f"{BASE}/2026/10/public_data_web_2026-09.csv",
}
URL_ESTACIONES = "https://gbfs.mex.lyftbikes.com/gbfs/en/station_information.json"

SUBRED = {
    "271-272": "Buenavista",
    "273-274": "Buenavista",
    "266-267": "Buenavista",
    "268-269": "Buenavista",
    "014": "Reforma",
    "474": "Reforma",
    "027": "Reforma",
    "018": "Reforma",
    "017": "Reforma",
    "015": "Reforma",
}

## 2. Descarga

Los archivos se guardan como `AAAA-MM.csv`, porque en el sitio tienen nombres distintos (`2025-10-1.csv`, `public_data_web_2026-08_2.csv`). Cada descarga se escribe primero en un archivo `.part`, para que un corte no deje un CSV a medias.

In [3]:
def descargar(url, ruta):
    """Descarga url en ruta si el archivo no existe."""
    if ruta.exists():
        return "ya existía"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_name(ruta.name + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(temporal, "wb") as f:
            for bloque in r.iter_content(chunk_size=1 << 20):
                f.write(bloque)
    temporal.rename(ruta)
    return "descargado"


rutas = {mes: DIR_TRIPS / f"{mes}.csv" for mes in URLS}
for mes, url in URLS.items():
    estado = descargar(url, rutas[mes])
    print(f"{mes}  {rutas[mes].stat().st_size / 1e6:7.1f} MB  {estado}")

print("estaciones:", descargar(URL_ESTACIONES, RUTA_ESTACIONES))

2025-10    107.6 MB  ya existía
2025-11     98.2 MB  ya existía
2025-12     88.5 MB  ya existía
2026-01     93.4 MB  ya existía
2026-02     93.7 MB  ya existía
2026-03    101.3 MB  ya existía
2026-04     97.8 MB  ya existía
2026-05     99.0 MB  ya existía
2026-06     96.1 MB  ya existía
2026-07     96.1 MB  ya existía
2026-08     94.8 MB  ya existía
2026-09     96.4 MB  ya existía
estaciones: ya existía


## 3. Carga del mes principal

Leemos septiembre completo y todo como texto, tal como viene en el CSV. No se quita ninguna fila.

La subred se marca con la columna booleana `toca_subred` (el retiro o el arribo es una de las 10 estaciones) en lugar de filtrar, porque para detectar los movimientos de camión hay que seguir cada bici aunque salga de la subred.

In [4]:
COL_RETIRO = "Ciclo_Estacion_Retiro"
COL_ARRIBO = "Ciclo_EstacionArribo"

df = pd.read_csv(rutas[MES_PRINCIPAL], dtype=str)
df["toca_subred"] = df[COL_RETIRO].isin(SUBRED) | df[COL_ARRIBO].isin(SUBRED)

estaciones = pd.DataFrame(json.loads(RUTA_ESTACIONES.read_text())["data"]["stations"])

print(f"Viajes en {MES_PRINCIPAL}: {len(df):,}")
print(f"Tocan la subred: {df['toca_subred'].sum():,} ({df['toca_subred'].mean():.1%})")
print(f"Estaciones en el feed: {len(estaciones)}")

Viajes en 2026-09: 1,498,934
Tocan la subred: 112,861 (7.5%)
Estaciones en el feed: 677


## 4. Vistazo a los datos crudos

Primero la forma general: primeras filas, columnas y nulos.

In [4]:
df.head()

,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_EstacionArribo,Fecha_Arribo,Hora_Arribo,toca_subred
0,M,30.0,6583322,028,31/08/2026,23:54:59,258,01/09/2026,00:00:03,False
1,F,33.0,4139655,030,31/08/2026,23:24:43,530,01/09/2026,00:00:04,False
2,O,32.0,6378079,139,31/08/2026,23:46:59,559,01/09/2026,00:00:35,False
3,M,35.0,6074272,365,31/08/2026,23:52:10,312,01/09/2026,00:00:44,False
4,M,30.0,4162376,412,31/08/2026,23:51:45,337,01/09/2026,00:00:55,False


In [5]:
pd.DataFrame({
    "nulos": df.isna().sum(),
    "valores_distintos": df.nunique(),
})

,nulos,valores_distintos
Genero_Usuario,0,4
Edad_Usuario,29,73
Bici,0,6574
Ciclo_Estacion_Retiro,0,676
Fecha_Retiro,0,38
Hora_Retiro,0,69839
Ciclo_EstacionArribo,0,676
Fecha_Arribo,0,30
Hora_Arribo,0,71072
toca_subred,0,2


Ahora las 10 estaciones de la subred: cuántos retiros y arribos registran en el mes, con sus docks según el feed. Son conteos crudos del mes completo, incluidos fines de semana y festivos.

In [6]:
info = estaciones.set_index("short_name")

resumen_subred = pd.DataFrame({
    "grupo": pd.Series(SUBRED),
    "nombre": info["name"].reindex(list(SUBRED)),
    "docks": info["capacity"].reindex(list(SUBRED)),
    "retiros": df[COL_RETIRO].value_counts().reindex(list(SUBRED), fill_value=0),
    "arribos": df[COL_ARRIBO].value_counts().reindex(list(SUBRED), fill_value=0),
})
resumen_subred

,grupo,nombre,docks,retiros,arribos
271-272,Buenavista,CE-271-272 Jesús García - Carlos J. Meneses,74,11270,14813
273-274,Buenavista,CE-273-274 Luis Donaldo Colosio - Av. Jesús Ga...,70,3395,4987
266-267,Buenavista,CE-266-267 Jesús García - Carlos J. Meneses,75,4400,7960
268-269,Buenavista,CE-268-269 Luis Donaldo Colosio - Av. Jesús Ga...,69,2033,2029
014,Reforma,CE-014 Reforma - Río de Plata,39,6172,7295
474,Reforma,CE-474 Bahía de San Hipólito-Bahía del Espírit...,31,3639,4048
027,Reforma,CE-027 Reforma- Havre,43,7719,8161
018,Reforma,CE-018 Reforma - Río Rhin,39,5977,6625
017,Reforma,CE-017 Reforma - Río Tiber,27,4708,5038
015,Reforma,CE-015 Reforma - Río Mississippi,23,4508,4487


## 5. Los 12 meses

Tabla de control para ver que todos los meses llegaron completos y con el mismo formato. Se lee un mes a la vez para no juntar todo en memoria. `mismas_columnas` compara cada archivo contra septiembre.

In [7]:
filas = []
for mes, ruta in rutas.items():
    m = pd.read_csv(ruta, dtype=str)
    toca = m[COL_RETIRO].isin(SUBRED) | m[COL_ARRIBO].isin(SUBRED)
    filas.append({
        "mes": mes,
        "viajes": len(m),
        "tocan_subred": int(toca.sum()),
        "pct_subred": round(100 * toca.mean(), 1),
        "mismas_columnas": list(m.columns) == list(df.columns[:-1]),
    })

resumen_meses = pd.DataFrame(filas).set_index("mes")
resumen_meses

,viajes,tocan_subred,pct_subred,mismas_columnas
mes,,,,
2025-10,1754353,142251,8.1,True
2025-11,1600741,126135,7.9,True
2025-12,1445068,98323,6.8,True
2026-01,1525302,107496,7.0,True
2026-02,1528610,119009,7.8,True
2026-03,1653373,129394,7.8,True
2026-04,1594967,124649,7.8,True
2026-05,1615360,117772,7.3,True
2026-06,1494265,124237,8.3,True


## 6. Fechas y horas a fecha-hora

Juntamos `Fecha` + `Hora` en una sola columna de fecha-hora para el retiro (`retiro`) y para el arribo (`arribo`). El formato es `dd/mm/aaaa hh:mm:ss` en los 12 meses. Con `errors="coerce"` lo que no se pueda leer queda como `NaT` en lugar de romper la celda, y lo contamos.

Todavía no se borra nada: solo vemos cuántas filas fallan y qué rango de fechas cubre el archivo.

In [ ]:
FORMATO = "%d/%m/%Y %H:%M:%S"

df["retiro"] = pd.to_datetime(df["Fecha_Retiro"] + " " + df["Hora_Retiro"], format=FORMATO, errors="coerce")
df["arribo"] = pd.to_datetime(df["Fecha_Arribo"] + " " + df["Hora_Arribo"], format=FORMATO, errors="coerce")

pd.DataFrame({
    "no_convertidas": df[["retiro", "arribo"]].isna().sum(),
    "minimo": df[["retiro", "arribo"]].min(),
    "maximo": df[["retiro", "arribo"]].max(),
})

In [ ]:
# Viajes cuyo retiro no cae en el mes principal (el CSV se corta por fecha de arribo)
fuera_de_mes = df["retiro"].dt.strftime("%Y-%m") != MES_PRINCIPAL
print(f"Retiros fuera de {MES_PRINCIPAL}: {fuera_de_mes.sum():,}")
df.loc[fuera_de_mes, "retiro"].dt.date.value_counts().sort_index()